> 在线讲义：[第 47 章 · 压缩与卸载：少存字节、轮流驻留，各付什么代价](https://wikieden.github.io/ai-infra-from-constraints/lessons/47-compress-offload.html) · 本书《从约束推导 AI Infra》

# 第 47 章 · 压缩与卸载：少存字节、轮流驻留，各付什么代价

配套讲义：仓库里 `ai-infra/notes/lessons/47-compress-offload.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。



Q2_K 位宽、M2 Max 容量、KV 的 q8_0 / q4_0、压缩的净节省、权重卸载与预取、压缩搬移的解压门槛取自 AII 第 8 章 §8.4。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
MiB, GiB = 2**20, 2**30

**函数 `group_bits`**：小工具。

In [ ]:
def group_bits(n: int, code_bits: int, meta_bytes: int) -> float:
    return (n * code_bits / 8 + meta_bytes) * 8 / n

**函数 `kv_mib`**：小工具。

In [ ]:
def kv_mib(base: float, fmt: str) -> float:
    return base * {"bf16": 64, "q8_0": 34, "q4_0": 18}[fmt] / 64

**函数 `net_saving_ms`**：小工具。

In [ ]:
def net_saving_ms(saved_mib: float, conv_ms: float, bw: float = 1.16e12) -> float:
    return saved_mib * MiB / bw * 1e3 - conv_ms

**函数 `offload`**：(腾出 MiB, 净省 MiB)。

In [ ]:
def offload(n_layers: int = 9, layer_mib: float = 288, buffers: int = 1) -> tuple[float, float]:
    """(腾出 MiB, 净省 MiB)。"""
    freed = n_layers * layer_mib
    return freed, freed - buffers * layer_mib

**函数 `copy_ms`**：小工具。

In [ ]:
def copy_ms(mib: float, bw: float) -> float:
    return mib * MiB / bw * 1e3

**函数 `min_decode_gbps`**：小工具。

In [ ]:
def min_decode_gbps(link: float, r: float) -> float:
    return link / (1 - r)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步：权重量化")
print(f"  Q2_K：{group_bits(256, 2, 20):.3f} bit/值，元数据占 {20 / 84:.0%}")
free = 96e9 / GiB - 8
print(f"  M2 Max 可用 {free:.2f} GiB：79.81 GiB 文件剩 {free - 79.81:.2f}，81.97 GiB 超 {81.97 - free:.2f}；放一条 8K（1.47 GiB）剩 {free - 79.81 - 1.47:.2f}")

In [ ]:
print("\n第二步：KV 压缩")
for f in ("q8_0", "q4_0"):
    print(f"  8K {f}：{kv_mib(1152, f):.0f} MiB（省 {1152 - kv_mib(1152, f):.0f}）")
print(f"  完整长请求 q8_0 {kv_mib(1188, 'q8_0'):.3f} MiB，16 条 {16 * kv_mib(1188, 'q8_0') / 1024:.2f} GiB")
for c in (0.2, 0.8):
    print(f"  少读 540 MiB、转换 {c} ms：净 {net_saving_ms(540, c):+.3f} ms")
print(f"  FP8 vs BF16 交点 {0.14 / 2e-5:.0f} token；2K {6.44 + 4.37e-5 * 2048:.2f} vs {6.58 + 2.37e-5 * 2048:.2f} ms，16K {6.44 + 4.37e-5 * 16384:.2f} vs {6.58 + 2.37e-5 * 16384:.2f} ms")

In [ ]:
print("\n第三步：卸载九份 FFN")
for buf in (1, 2):
    freed, net = offload(buffers=buf)
    print(f"  {buf} 组预取缓冲：净省 {net:.0f} MiB → 多放 {int(net // 1152)} 条 8K 输入 / {int(net // 1188)} 条完整请求")
for name, bw in (("PCIe Gen5", 64e9), ("NVLink-C2C", 450e9)):
    t = copy_ms(2592, bw)
    print(f"  {name}：每轮 {t:.1f} ms，255 轮 {255 * t / 1e3:.2f} s")
print(f"  batch 4 上限 {4 / 0.0425:.0f} token/s")
print(f"  压到 1/4 再传：96 MiB 直传 {copy_ms(96, 64e9):.2f} ms、压缩后 {copy_ms(24, 64e9):.2f} ms；解压要 > {min_decode_gbps(64, 0.25):.1f} GB/s")

## 3. 代码里才看得到的

1. M2 Max 的 96 GB 是十进制，换成 89.41 GiB 再减 8 GiB。
2. `net_saving_ms` 代码得 +0.288 / −0.312，原文写 +0.287 / −0.313：带宽取整差异。
3. 2592 MiB = 2.718 GB，所以 PCIe Gen5 下是 42.5 ms 而不是 40.5 ms。

## 4. 自己改一改

- 给 `kv_mib` 加 FP8（每 128 个值一个 FP32 scale）格式。
- 用 `offload` 找 PCIe Gen5 下每轮搬移不超过 26 ms 时最多能卸几层。
- 把输出间隔拟合式扫到 128K，画 BF16 和 FP8 的差。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 照片压缩每块还要记基准，位越低基准越显眼；
> 包装压得小省车厢，拆包慢了白省；上下文越长越划算；
> 借邻居冰箱，每轮都去取，路远就来不及；
> 换速记法逐题对，错的题会换人。